# 04. Was steckt in 63 richtigen Vorhersagen?

**Leitfrage:** Wie gut erkennt kNN positive und negative Filmkritiken?

**Demonstration:** [Testurteile zählen und die Konfusionsmatrix prüfen](#counts)

**Dein Arbeitsauftrag:** [Drei Kennzahlen implementieren](#scores) und [beide Klassen vergleichen](#classes).

In [ ]:
%pip install scikit-learn==1.9.1

<a id='counts'></a>

## 1. Demonstration – vom Testurteil zur Konfusionsmatrix

### 1.1 Zwei Urteile lesen

Die [Vorhersagen des Referenzmodells](data/imdb/reference_test_predictions.csv) stammen aus dem Testlauf von Notebook 03 mit sieben Nachbarn und Kosinusdistanz. Jede Zeile enthält den Dateinamen (`file_name`), das tatsächliche Urteil (`actual`) und die Vorhersage (`predicted`) für eine der 100 Reviews. Hier kannst du mit einem frischen Kernel starten. Für die Beispiele auf den Folien ist `reference_test_predictions.csv` eingestellt. Für deinen eigenen Testlauf aus [Notebook 03](03_model_selection.ipynb) ändere unten den Dateinamen auf `my_test_predictions.csv`.

In [ ]:
import csv
from pathlib import Path

data_dir = Path('data/imdb')
predictions_file = data_dir / 'reference_test_predictions.csv'

y_true = []
y_pred = []
with predictions_file.open(encoding='utf-8', newline='') as stream:
    for row in csv.DictReader(stream):
        y_true.append(row['actual'])
        y_pred.append(row['predicted'])

In [ ]:
print('Actual  Predicted')
for actual, predicted in zip(y_true[:2], y_pred[:2]):
    print(f'{actual:7} {predicted}')

Die Zielklasse ist `pos`. Welche Vorhersagen sind richtig? Wird eine positive Kritik übersehen?

### 1.2 Vier Fälle zählen

Gehe die beiden Entscheidungen der Schleife durch: Stimmt die Vorhersage? Gehört die Review tatsächlich zur Zielklasse? Ersetze gemeinsam mit der Gruppe jedes `pass` durch die passende Zählerzeile. Die Funktion gibt TP, FP, FN und TN in dieser Reihenfolge zurück.

In [ ]:
def count_cases(y_true: list[str], y_pred: list[str], target: str) -> tuple[int, int, int, int]:
    tp = fp = fn = tn = 0

    for actual, predicted in zip(y_true, y_pred):
        if actual == predicted:
            if actual == target:
                pass
            else:
                pass
        else:
            if actual == target:
                pass
            else:
                pass

    return tp, fp, fn, tn

In [ ]:
tp, fp, fn, tn = count_cases(y_true, y_pred, target='pos')
print(f'TP: {tp} | FP: {fp} | FN: {fn} | TN: {tn}')
print(f'Correct: {tp + tn} of {len(y_true)}')

### 1.3 Gegenprobe mit scikit-learn

Die Zeilen zeigen das tatsächliche Urteil, die Spalten die Vorhersage. Mit `labels=['pos', 'neg']` steht TP oben links. Wo findest du die beiden Fehlertypen in der Matrix wieder? [Dokumentation zu confusion_matrix](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.confusion_matrix.html)

In [ ]:
from sklearn.metrics import confusion_matrix

matrix = confusion_matrix(y_true, y_pred, labels=['pos', 'neg'])
print(matrix)
assert matrix.tolist() == [[tp, fn], [fp, tn]]

<a id='scores'></a>

## 2. Arbeitsauftrag – Precision, Recall und F₁ implementieren

Übersetze jede Formel von den Folien in eine Rückgabezeile. Verwende Anteile zwischen 0 und 1 und lasse die Selbsttests laufen. Für einen Nenner von null verwenden wir in dieser Übung den Wert 0.

### 2.1 Precision

Von allen Meldungen für die Zielklasse: Welcher Anteil ist richtig?

In [ ]:
def precision(tp: int, fp: int) -> float:
    if tp + fp == 0:
        return 0.0
    return ...  # TODO: Calculate precision.

In [ ]:
assert precision(3, 1) == 0.75
assert precision(0, 0) == 0.0

### 2.2 Recall

Von allen Reviews, die tatsächlich zur Zielklasse gehören: Welchen Anteil findet das Modell?

In [ ]:
def recall(tp: int, fn: int) -> float:
    if tp + fn == 0:
        return 0.0
    return ...  # TODO: Calculate recall.

In [ ]:
assert recall(3, 3) == 0.5
assert recall(0, 0) == 0.0

### 2.3 F₁-Score

Berechne das harmonische Mittel aus Precision und Recall.

In [ ]:
def f1_score(precision: float, recall: float) -> float:
    if precision + recall == 0:
        return 0.0
    return ...  # TODO: Calculate f1_score.

In [ ]:
from math import isclose

assert isclose(f1_score(0.9, 0.1), 0.18)
assert f1_score(0.0, 0.0) == 0.0

<a id='classes'></a>

## 3. Die Zielklasse wechseln

### 3.1 Eigene Kennzahlen vergleichen

Führe die vorbereitete Auswertung mit deinen Funktionen aus. Die Vorhersagen bleiben gleich. Was zählt jetzt als Treffer, wenn `neg` die Zielklasse ist?

In [ ]:
print('Target   Precision   Recall     F1')
for target in ['pos', 'neg']:
    tp, fp, fn, tn = count_cases(y_true, y_pred, target)
    p = precision(tp, fp)
    r = recall(tp, fn)
    print(f'{target:7} {p:10.2f} {r:8.2f} {f1_score(p, r):6.2f}')

### 3.2 Gegenprobe mit dem Classification Report

Vergleiche die Zeilen `pos` und `neg` mit deiner Tabelle. Die Spalte `support` zählt die tatsächlichen Reviews der jeweiligen Klasse. [Dokumentation zu classification_report](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.classification_report.html)

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(y_true, y_pred, labels=['pos', 'neg']))

Beide Klassen enthalten 50 Reviews. Warum unterscheiden sich ihre Recall-Werte? Begründe deine Antwort mit den Trefferzahlen aus der Konfusionsmatrix.